# SecureByDesign — Pipeline Notebook (Person A)
## Explainable LLM-Based STRIDE Threat Inference — Groq Backend

**LLM:** Groq `llama-3.3-70b-versatile` (free tier) — get key at https://console.groq.com/keys

**Before running:** Add `GROQ_API_KEY` to Kaggle Secrets + enable Internet toggle.

**Run all cells top-to-bottom.**

# PHASE 1: Environment Setup

In [ ]:
# Install dependencies — groq replaces google-generativeai
!pip install groq scikit-learn pandas python-dateutil --quiet

import groq, sklearn, pandas as pd
print(f'✅ groq: {groq.__version__}')
print(f'✅ scikit-learn: {sklearn.__version__}')
print(f'✅ pandas: {pd.__version__}')

In [ ]:
# Configure Groq API key from Kaggle Secrets
# Step: Notebook editor → Add-ons → Secrets → Add GROQ_API_KEY
# Get free key: https://console.groq.com/keys
from kaggle_secrets import UserSecretsClient
from groq import Groq

try:
    GROQ_API_KEY = UserSecretsClient().get_secret('GROQ_API_KEY')
    print('✅ Kaggle Secret GROQ_API_KEY loaded')
except Exception as e:
    print(f'⚠ Kaggle Secret not found: {e}')
    GROQ_API_KEY = input('Enter your Groq API key: ')

# Test connection
client = Groq(api_key=GROQ_API_KEY)
resp = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role':'user','content':'Say exactly: API_CONNECTION_SUCCESSFUL'}],
    max_tokens=20
)
print(f'✅ API Test: {resp.choices[0].message.content.strip()}')

In [ ]:
import os, sys

BASE = '/kaggle/working/SecureByDesign'
for d in ['pipeline','evaluation/results','evaluation/test_dfds','app','data']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

os.chdir(BASE)
sys.path.insert(0, BASE)
print(f'✅ Working directory: {os.getcwd()}')
print(f'✅ sys.path: {BASE}')

In [ ]:
# Clone microSecEnD dataset
DATA_DIR = f'{BASE}/data/microSecEnD'
if not os.path.exists(DATA_DIR):
    !git clone https://github.com/tuhh-softsec/microSecEnD {DATA_DIR} --quiet
    print('✅ microSecEnD cloned')
else:
    print('✅ microSecEnD already present')
print(f'Dataset items: {len(os.listdir(DATA_DIR))}')

# PHASE 2: DFD Parser

In [ ]:
%%writefile /kaggle/working/SecureByDesign/pipeline/dfd_parser.py
"""
DFD Parser for SecureByDesign. Parses DFD JSON into structured context for LLM.
Author: Person A
"""
from typing import Dict, List
from dataclasses import dataclass, field

@dataclass
class ParsedDFD:
    dfd_id: str; system_name: str
    external_entities: List[Dict] = field(default_factory=list)
    processes: List[Dict] = field(default_factory=list)
    datastores: List[Dict] = field(default_factory=list)
    edges: List[Dict] = field(default_factory=list)
    trust_boundaries: List[Dict] = field(default_factory=list)
    boundary_crossing_edges: List[Dict] = field(default_factory=list)
    missing_auth_edges: List[str] = field(default_factory=list)
    missing_encryption_edges: List[str] = field(default_factory=list)
    unknown_protocol_edges: List[str] = field(default_factory=list)
    is_partial: bool = False
    completeness_score: float = 1.0
    missing_elements: List[str] = field(default_factory=list)
    text_summary: str = ''

ParsedDFD.nodes_all = property(lambda self: self.external_entities+self.processes+self.datastores)

def parse_dfd(dfd_json: dict) -> ParsedDFD:
    for f in ['dfd_id','system_name','nodes','edges']:
        if f not in dfd_json: raise ValueError(f'Missing required field: {f}')
    p = ParsedDFD(dfd_id=dfd_json['dfd_id'], system_name=dfd_json['system_name'])
    nl = {n['id']:n for n in dfd_json.get('nodes',[]) if 'id' in n}
    for node in dfd_json.get('nodes',[]):
        t = node.get('type','').lower()
        if t=='external_entity': p.external_entities.append(node)
        elif t=='datastore': p.datastores.append(node)
        else: p.processes.append(node)
    p.edges = dfd_json.get('edges',[])
    for e in p.edges:
        i = e.get('id','?')
        if e.get('authenticated') is None: p.missing_auth_edges.append(i)
        if e.get('encrypted') is None: p.missing_encryption_edges.append(i)
        if e.get('protocol') is None: p.unknown_protocol_edges.append(i)
    p.trust_boundaries = dfd_json.get('trust_boundaries',[])
    for e in p.edges:
        fn,tn = e.get('from'),e.get('to')
        for tb in p.trust_boundaries:
            sep = set(tb.get('separates',[]))
            if (fn in sep) != (tn in sep):
                p.boundary_crossing_edges.append({**e,'crossing_boundary':tb.get('id'),'boundary_name':tb.get('name','?'),'from_name':nl.get(fn,{}).get('name',fn),'to_name':nl.get(tn,{}).get('name',tn)})
                break
    pf = dfd_json.get('partial_info_flags',{}); miss=[]; fac=[]
    if not p.trust_boundaries or pf.get('missing_trust_boundaries'):
        miss.append('Trust boundaries not defined'); fac.append(0.0)
    else: fac.append(1.0)
    n = max(len(p.edges),1)
    ar = 1.0-len(p.missing_auth_edges)/n; fac.append(ar)
    if p.missing_auth_edges: miss.append(f'Auth unspecified: {", ".join(p.missing_auth_edges)}')
    er = 1.0-len(p.missing_encryption_edges)/n; fac.append(er)
    if p.missing_encryption_edges: miss.append(f'Enc unspecified: {", ".join(p.missing_encryption_edges)}')
    if pf.get('unknown_protocols') or p.unknown_protocol_edges:
        miss.append(f'Protocol unknown: {", ".join(p.unknown_protocol_edges)}'); fac.append(0.5)
    else: fac.append(1.0)
    if not p.nodes_all: miss.append('No nodes'); fac.append(0.0)
    if not p.edges: miss.append('No edges'); fac.append(0.0)
    p.completeness_score = sum(fac)/max(len(fac),1)
    p.is_partial = p.completeness_score<0.8 or bool(miss)
    p.missing_elements = miss
    p.text_summary = _summary(p, nl)
    return p

def _summary(p: ParsedDFD, nl: dict) -> str:
    L=[f'SYSTEM: {p.system_name}',f'DFD ID: {p.dfd_id}',f'COMPLETENESS: {p.completeness_score:.0%}','']
    L.append('=== COMPONENTS ===')
    if p.external_entities: L.append('External Entities: '+', '.join(n['name'] for n in p.external_entities))
    if p.processes: L.append('Processes: '+', '.join(n['name'] for n in p.processes))
    if p.datastores: L.append('Datastores: '+', '.join(n['name'] for n in p.datastores))
    L+=['','=== TRUST BOUNDARIES ===']
    if p.trust_boundaries:
        for tb in p.trust_boundaries:
            names=[nl.get(x,{}).get('name',x) for x in tb.get('separates',[])]
            L.append(f'  [{tb["id"]}] {tb.get("name","?")} : separates {" | ".join(names)}')
    else: L.append('  WARNING: No trust boundaries defined.')
    L+=['','=== DATA FLOWS ===']
    for e in p.edges:
        fn=nl.get(e.get('from'),{}).get('name',e.get('from','?'))
        tn=nl.get(e.get('to'),{}).get('name',e.get('to','?'))
        a=e.get('authenticated'); enc=e.get('encrypted')
        astr='authenticated' if a is True else 'NOT authenticated' if a is False else 'AUTHENTICATION UNSPECIFIED'
        estr='encrypted' if enc is True else 'NOT encrypted' if enc is False else 'ENCRYPTION UNSPECIFIED'
        L+=[f'  [{e.get("id","?")}] {fn} \u2192 {tn}',f'       Data: {e.get("data_description","?")} | Protocol: {e.get("protocol") or "UNKNOWN"} | {astr} | {estr}']
    L.append('')
    if p.boundary_crossing_edges:
        L.append('=== TRUST BOUNDARY CROSSINGS (HIGH SECURITY RELEVANCE) ===')
        for c in p.boundary_crossing_edges:
            a=c.get('authenticated'); enc=c.get('encrypted')
            L+=[f'  [{c.get("id","?")}] {c.get("from_name","?")} \u2192 {c.get("to_name","?")}',f'       Crosses: {c.get("boundary_name","?")} | Auth: {"UNSPECIFIED" if a is None else a} | Enc: {"UNSPECIFIED" if enc is None else enc}']
        L.append('')
    if p.is_partial:
        L.append('=== PARTIAL DFD WARNINGS ===')
        for w in p.missing_elements: L.append(f'  \u26a0  {w}')
        L.append('')
    return '\n'.join(L)

In [ ]:
from pipeline.dfd_parser import parse_dfd
sample = {
    'dfd_id':'t1','system_name':'Test Service',
    'nodes':[{'id':'N1','type':'external_entity','name':'Client'},{'id':'N2','type':'process','name':'API'},{'id':'N3','type':'datastore','name':'DB'}],
    'edges':[{'id':'E1','from':'N1','to':'N2','data_description':'Login','protocol':'HTTPS','authenticated':None,'encrypted':True},
             {'id':'E2','from':'N2','to':'N3','data_description':'Query','protocol':None,'authenticated':None,'encrypted':None}],
    'trust_boundaries':[{'id':'TB1','name':'Internet','separates':['N1','N2']}],
    'partial_info_flags':{'missing_trust_boundaries':False,'unknown_protocols':True,'unspecified_auth':True,'incomplete_nodes':False}
}
r = parse_dfd(sample)
assert r.completeness_score < 1.0 and r.is_partial
assert len(r.boundary_crossing_edges) == 1
print(f'✅ Completeness: {r.completeness_score:.0%}, Partial: {r.is_partial}, Crossings: {len(r.boundary_crossing_edges)}')
print(r.text_summary)
print('✅ PHASE 2 COMPLETE')

# PHASE 3: Prompt Engineering

Upload `pipeline/prompt_templates.py` as a Kaggle Dataset, then attach it. The cell below checks for it. If missing, paste the file content as instructed.

In [ ]:
import shutil, os

DST = '/kaggle/working/SecureByDesign/pipeline/prompt_templates.py'
# Walk all input dataset paths to find the file
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'prompt_templates.py':
            shutil.copy(os.path.join(root, f), DST)
            print(f'✅ Copied from {root}')
            break

if os.path.exists(DST) and os.path.getsize(DST) > 1000:
    print(f'✅ prompt_templates.py ready ({os.path.getsize(DST):,} bytes)')
else:
    raise FileNotFoundError(
        'Upload pipeline/prompt_templates.py as a Kaggle Dataset and attach it to this notebook.\n'
        'Kaggle: + Add data → Your datasets → Upload → attach'

In [ ]:
from pipeline.prompt_templates import build_analysis_prompt, SYSTEM_PROMPT
msgs = build_analysis_prompt('SYSTEM: Test\nDFD ID: t1\nCOMPLETENESS: 50%', 'internet-facing')
assert len(msgs) == 7 and msgs[-1]['role'] == 'user'
print(f'✅ Messages: {len(msgs)}, Final role: {msgs[-1]["role"]}')
print(f'✅ System prompt: {len(SYSTEM_PROMPT):,} chars')
print('✅ PHASE 3 COMPLETE')

# PHASE 4: Response Parser

In [ ]:
%%writefile /kaggle/working/SecureByDesign/pipeline/response_parser.py
"""
Response Parser for SecureByDesign. Parses/validates LLM JSON output. Never raises.
Author: Person A
"""
import json, re
from datetime import datetime
from typing import Optional, Any

VALID_STRIDE={'Spoofing','Tampering','Repudiation','Information Disclosure','Denial of Service','Elevation of Privilege'}
VALID_CONF={'High','Medium','Low'}
VALID_RISK={'Critical','High','Medium','Low'}
ALIASES={'spoof':'Spoofing','tamper':'Tampering','repudiat':'Repudiation','disclosure':'Information Disclosure','denial':'Denial of Service','dos':'Denial of Service','privilege':'Elevation of Privilege','escalation':'Elevation of Privilege','eop':'Elevation of Privilege'}

def parse_llm_response(raw:str, dfd_id:str, system_name:str)->dict:
    js=_xjson(raw)
    if js is None: return _err(dfd_id,system_name,f'No JSON found. Got: {str(raw)[:80]}')
    try: r=json.loads(js)
    except json.JSONDecodeError as e:
        try: r=json.loads(re.sub(r',\s*([}\]])',r'\1',js))
        except: return _err(dfd_id,system_name,f'JSON parse error: {e}')
    if not isinstance(r,dict): return _err(dfd_id,system_name,'LLM returned array not object')
    return _norm(r,dfd_id,system_name)

def _xjson(text:str)->Optional[str]:
    if not text: return None
    text=text.strip()
    if text.startswith('{'): return text
    for pat in [r'```json\s*([\s\S]*?)\s*```',r'```\s*([\s\S]*?)\s*```',r'`([\s\S]*?)`']:
        m=re.search(pat,text)
        if m:
            c=m.group(1).strip()
            if c.startswith('{'): return c
    start=text.find('{')
    if start==-1: return None
    depth,in_s,esc=0,False,False
    for i,ch in enumerate(text[start:],start):
        if esc: esc=False; continue
        if ch=='\\': esc=True; continue
        if ch=='"': in_s=not in_s; continue
        if in_s: continue
        if ch=='{': depth+=1
        elif ch=='}':
            depth-=1
            if depth==0: return text[start:i+1]
    return None

def _norm(r:dict,dfd_id:str,system_name:str)->dict:
    r.setdefault('dfd_id',dfd_id); r.setdefault('system_name',system_name)
    r.setdefault('analysis_timestamp',datetime.utcnow().isoformat()+'Z')
    r.setdefault('partial_dfd_detected',False); r.setdefault('missing_controls_summary',[])
    if r.get('overall_risk_level') not in VALID_RISK: r['overall_risk_level']='High'
    rt=r.get('threats',[])
    if not isinstance(rt,list): rt=[]
    r['threats']=[t for i,th in enumerate(rt) if (t:=_nt(th,i+1)) is not None]
    cov={c:0 for c in VALID_STRIDE}
    for t in r['threats']:
        if t.get('stride_category') in cov: cov[t['stride_category']]+=1
    r['stride_coverage']=cov
    return r

def _nt(t:Any,idx:int)->Optional[dict]:
    if not isinstance(t,dict): return None
    t.setdefault('threat_id',f'T{idx}'); t.setdefault('affected_component','Unspecified')
    t.setdefault('threat_description','Not provided'); t.setdefault('missing_control','Not specified')
    t.setdefault('confidence_reason','Not specified'); t.setdefault('explanation','Not provided')
    raw=str(t.get('stride_category','')).strip()
    if raw not in VALID_STRIDE:
        rl=raw.lower(); matched=None
        for alias,canon in ALIASES.items():
            if alias in rl: matched=canon; break
        if not matched:
            for c in VALID_STRIDE:
                if c.lower() in rl or rl in c.lower(): matched=c; break
        t['stride_category']=matched or 'Information Disclosure'
    c=str(t.get('confidence','')).strip().capitalize()
    t['confidence']=c if c in VALID_CONF else 'Low'
    return t

def _err(dfd_id,system_name,msg)->dict:
    return {'dfd_id':dfd_id,'system_name':system_name,'analysis_timestamp':datetime.utcnow().isoformat()+'Z',
            'overall_risk_level':'Unknown','partial_dfd_detected':True,'error':msg,'threats':[],
            'missing_controls_summary':[f'Analysis failed: {msg}'],
            'stride_coverage':{c:0 for c in VALID_STRIDE}}

In [ ]:
import json
from pipeline.response_parser import parse_llm_response

good = json.dumps({'overall_risk_level':'High','threats':[{'stride_category':'Spoofing','confidence':'High','affected_component':'E1','threat_description':'x','missing_control':'y','confidence_reason':'z','explanation':'w'}]})
r1 = parse_llm_response(good,'t1','Test'); assert len(r1['threats'])==1
print('✅ Test 1: Clean JSON')
r2 = parse_llm_response(f'```json\n{good}\n```','t1','Test'); assert len(r2['threats'])==1
print('✅ Test 2: Markdown-wrapped JSON')
r3 = parse_llm_response('sorry cannot help','t1','Test'); assert 'error' in r3
print('✅ Test 3: Garbage input handled')
r4 = parse_llm_response(json.dumps({'threats':[{'stride_category':'privilege escalation','confidence':'High','affected_component':'E1','threat_description':'x','missing_control':'y','confidence_reason':'z','explanation':'w'}]}),'t2','Test')
assert r4['threats'][0]['stride_category']=='Elevation of Privilege'
print(f"✅ Test 4: Fuzzy STRIDE → '{r4['threats'][0]['stride_category']}'")
print('✅ PHASE 4 COMPLETE')

# PHASE 5: Inference Engine — `analyze_dfd()` via Groq

In [ ]:
%%writefile /kaggle/working/SecureByDesign/pipeline/inference.py
"""
Inference Engine for SecureByDesign.
LLM: Groq llama-3.3-70b-versatile (free tier). Never raises. Contract: analyze_dfd().
Author: Person A
"""
import time, os
from datetime import datetime
from typing import Optional
from groq import Groq
from pipeline.dfd_parser import parse_dfd
from pipeline.prompt_templates import SYSTEM_PROMPT, build_analysis_prompt
from pipeline.response_parser import parse_llm_response

MODEL = 'llama-3.3-70b-versatile'
MAX_RETRIES = 3; RETRY_DELAY = 2; MAX_TOKENS = 4096; TEMP = 0.1
CATS = ['Spoofing','Tampering','Repudiation','Information Disclosure','Denial of Service','Elevation of Privilege']

def _key()->str:
    try:
        from kaggle_secrets import UserSecretsClient
        k=UserSecretsClient().get_secret('GROQ_API_KEY')
        if k: return k
    except: pass
    return os.environ.get('GROQ_API_KEY','')

def _to_groq(msgs:list)->list:
    """Convert Gemini-format messages [{role,parts}] → Groq-format [{role,content}]."""
    out=[]
    for m in msgs:
        role='assistant' if m['role']=='model' else m['role']
        content=m['parts'][0] if isinstance(m.get('parts'),list) else m.get('content','')
        out.append({'role':role,'content':content})
    return out

def analyze_dfd(dfd_json:dict, security_context:str='')->dict:
    """
    Person B's contract: analyze a DFD JSON → STRIDE threat report dict.
    Always returns schema-compliant dict. Never raises.
    """
    t0=time.time()
    did=dfd_json.get('dfd_id','unknown'); sn=dfd_json.get('system_name','Unknown')
    print(f'[SbD] Analyzing: {did} | model={MODEL}')
    try:
        parsed=parse_dfd(dfd_json)
        print(f'[SbD] parsed: completeness={parsed.completeness_score:.0%} partial={parsed.is_partial}')
    except ValueError as e:
        return _err(did,sn,f'DFD parse error: {e}')
    msgs=_to_groq(build_analysis_prompt(parsed.text_summary,security_context))
    msgs.insert(0,{'role':'system','content':SYSTEM_PROMPT})
    raw=None; last_e=None
    client=Groq(api_key=_key())
    for attempt in range(1,MAX_RETRIES+1):
        try:
            resp=client.chat.completions.create(model=MODEL,messages=msgs,temperature=TEMP,max_tokens=MAX_TOKENS,response_format={'type':'json_object'})
            raw=resp.choices[0].message.content
            u=resp.usage
            print(f'[SbD] OK attempt={attempt} len={len(raw)} in={u.prompt_tokens} out={u.completion_tokens}')
            break
        except Exception as e:
            last_e=str(e); print(f'[SbD] fail {attempt}/{MAX_RETRIES}: {last_e[:100]}')
            if attempt<MAX_RETRIES: time.sleep(RETRY_DELAY*attempt)
    if raw is None:
        return _err(did,sn,f'Groq failed {MAX_RETRIES} attempts: {last_e}',parsed.completeness_score,parsed.is_partial,parsed.missing_elements)
    result=parse_llm_response(raw,did,sn)
    result.update({'partial_dfd_detected':parsed.is_partial,'completeness_score':round(parsed.completeness_score,3),
                   'dfd_missing_elements':parsed.missing_elements,'analysis_duration_seconds':round(time.time()-t0,2),'model_used':MODEL})
    print(f'[SbD] done: threats={len(result.get("threats",[]))} risk={result.get("overall_risk_level")} dur={result["analysis_duration_seconds"]}s')
    return result

def _err(did,sn,msg,cs=0.0,partial=True,missing=None)->dict:
    return {'dfd_id':did,'system_name':sn,'analysis_timestamp':datetime.utcnow().isoformat()+'Z',
            'overall_risk_level':'Unknown','partial_dfd_detected':partial,'error':msg,'threats':[],
            'missing_controls_summary':[f'Failed: {msg}'],'stride_coverage':{c:0 for c in CATS},
            'completeness_score':cs,'dfd_missing_elements':missing or [],'analysis_duration_seconds':0.0,'model_used':MODEL}

In [ ]:
%%writefile /kaggle/working/SecureByDesign/pipeline/__init__.py
from pipeline.inference import analyze_dfd
__all__ = ['analyze_dfd']

In [ ]:
from pipeline.inference import analyze_dfd

test_dfd = {
    'dfd_id':'integration_001','system_name':'User Auth Microservice',
    'nodes':[
        {'id':'N1','type':'external_entity','name':'Mobile Client'},
        {'id':'N2','type':'process','name':'API Gateway'},
        {'id':'N3','type':'process','name':'Auth Service'},
        {'id':'N4','type':'datastore','name':'User Database'}
    ],
    'edges':[
        {'id':'E1','from':'N1','to':'N2','data_description':'Login credentials','protocol':'HTTPS','authenticated':None,'encrypted':True},
        {'id':'E2','from':'N2','to':'N3','data_description':'Auth request','protocol':'HTTP','authenticated':False,'encrypted':False},
        {'id':'E3','from':'N3','to':'N4','data_description':'User lookup','protocol':'TCP','authenticated':True,'encrypted':None}
    ],
    'trust_boundaries':[
        {'id':'TB1','name':'Internet Boundary','separates':['N1','N2']},
        {'id':'TB2','name':'Internal Service Boundary','separates':['N2','N3']}
    ],
    'partial_info_flags':{'missing_trust_boundaries':False,'unknown_protocols':False,'unspecified_auth':True,'incomplete_nodes':False}
}

result = analyze_dfd(test_dfd, 'Internet-facing fintech auth service. Handles login and JWT issuance.')

print('\n=== INTEGRATION TEST RESULTS ===')
print(f"Risk          : {result.get('overall_risk_level')}")
print(f"Threats Found : {len(result.get('threats',[]))}")
print(f"Partial DFD   : {result.get('partial_dfd_detected')}")
print(f"Completeness  : {result.get('completeness_score',0):.0%}")
print(f"Duration      : {result.get('analysis_duration_seconds')}s")
print(f"Model         : {result.get('model_used')}")
print(f"STRIDE        : {result.get('stride_coverage',{})}")

if result.get('error'):
    print(f"\n⚠ Error: {result['error']}")
elif result.get('threats'):
    t=result['threats'][0]
    print(f"\nTop Threat: [{t['stride_category']}] {t['affected_component']}")
    print(f"Confidence: {t['confidence']} — {t['confidence_reason']}")
    print(f"{t['explanation'][:250]}")

assert 'threats' in result and 'stride_coverage' in result
print('\n✅ PHASE 5 COMPLETE')

# PHASE 6: Partial DFD Stress Test (Novel Contribution)

In [ ]:
import pandas as pd, copy, time, os
from pipeline.inference import analyze_dfd

base_dfd = {
    'dfd_id':'deg_base','system_name':'Order Processing Service',
    'nodes':[
        {'id':'N1','type':'external_entity','name':'Web Client'},
        {'id':'N2','type':'process','name':'Order API'},
        {'id':'N3','type':'process','name':'Payment Service'},
        {'id':'N4','type':'datastore','name':'Orders DB'}
    ],
    'edges':[
        {'id':'E1','from':'N1','to':'N2','data_description':'Order+payment','protocol':'HTTPS','authenticated':True,'encrypted':True},
        {'id':'E2','from':'N2','to':'N3','data_description':'Card data','protocol':'HTTPS','authenticated':True,'encrypted':True},
        {'id':'E3','from':'N2','to':'N4','data_description':'Order record','protocol':'TCP','authenticated':True,'encrypted':True},
    ],
    'trust_boundaries':[{'id':'TB1','name':'Internet Boundary','separates':['N1','N2']}],
    'partial_info_flags':{'missing_trust_boundaries':False,'unknown_protocols':False,'unspecified_auth':False,'incomplete_nodes':False}
}

scenarios = {
    '1-Full DFD (100%)': base_dfd,
    '2-No Trust Boundaries': {**base_dfd,'trust_boundaries':[]},
    '3-No Auth Info': {**base_dfd,'edges':[{**e,'authenticated':None} for e in base_dfd['edges']]},
    '4-No Encryption Info': {**base_dfd,'edges':[{**e,'encrypted':None} for e in base_dfd['edges']]},
    '5-Minimal (No TB+Auth+Enc+Protocol)': {
        **base_dfd,'trust_boundaries':[],
        'edges':[{**e,'authenticated':None,'encrypted':None,'protocol':None} for e in base_dfd['edges']]
    }
}

rows=[]
for name, dfd in scenarios.items():
    d = copy.deepcopy(dfd)
    d['partial_info_flags'] = {
        'missing_trust_boundaries': len(d.get('trust_boundaries',[])) == 0,
        'unknown_protocols': any(e.get('protocol') is None for e in d.get('edges',[])),
        'unspecified_auth': any(e.get('authenticated') is None for e in d.get('edges',[])),
        'incomplete_nodes': False
    }
    d['dfd_id'] = f'deg_{name[:4].strip()}'
    print(f'\n--- Scenario: {name} ---')
    r = analyze_dfd(d)
    threats = r.get('threats',[])
    conf = {'High':0,'Medium':0,'Low':0}
    for t in threats: conf[t.get('confidence','Low')] = conf.get(t.get('confidence','Low'),0)+1
    rows.append({'Scenario':name,'Threats':len(threats),'High':conf['High'],'Medium':conf['Medium'],'Low':conf['Low'],
                 'Completeness':f"{r.get('completeness_score',0):.0%}",'Partial':r.get('partial_dfd_detected',False)})
    time.sleep(1)  # Rate limit courtesy delay

df = pd.DataFrame(rows)
print('\n=== DEGRADATION EXPERIMENT RESULTS ===')
print(df.to_string(index=False))

os.makedirs('/kaggle/working/SecureByDesign/evaluation/results', exist_ok=True)
df.to_csv('/kaggle/working/SecureByDesign/evaluation/results/degradation_experiment.csv', index=False)
print('\n✅ CSV saved: evaluation/results/degradation_experiment.csv')
print('✅ PHASE 6 COMPLETE')

# PHASE 7: Handoff Verification & GitHub Push

In [ ]:
import os
from pipeline.inference import analyze_dfd
from pipeline.response_parser import parse_llm_response

print('=== HANDOFF VERIFICATION ===')

# 1. Import
try:
    from pipeline.inference import analyze_dfd
    print('✅ from pipeline.inference import analyze_dfd')
except Exception as e: print(f'❌ {e}')

# 2. Valid DFD produces schema-compliant output
r = analyze_dfd({'dfd_id':'hc','system_name':'Handoff Check','nodes':[{'id':'N1','type':'process','name':'S'},{'id':'N2','type':'datastore','name':'D'}],
                 'edges':[{'id':'E1','from':'N1','to':'N2','data_description':'q','protocol':None,'authenticated':None,'encrypted':None}],
                 'trust_boundaries':[],'partial_info_flags':{'missing_trust_boundaries':True,'unknown_protocols':True,'unspecified_auth':True,'incomplete_nodes':False}})
required = ['dfd_id','system_name','analysis_timestamp','overall_risk_level','partial_dfd_detected','threats','missing_controls_summary','stride_coverage']
missing = [k for k in required if k not in r]
print(f'✅ Schema fields: {len(required)-len(missing)}/{len(required)} present' if not missing else f'❌ Missing: {missing}')

# 3. Bad input → graceful error
bad = analyze_dfd({'system_name':'bad'}); assert 'error' in bad or 'threats' in bad
print('✅ Bad input: graceful error (no crash)')

# 4. Parser edge cases
assert 'error' in parse_llm_response('', 'x', 'y')
assert 'error' in parse_llm_response('I cannot help', 'x', 'y')
print('✅ Response parser edge cases: handled')

# 5. CSV
csv = '/kaggle/working/SecureByDesign/evaluation/results/degradation_experiment.csv'
print(f'✅ Degradation CSV: {os.path.getsize(csv)} bytes' if os.path.exists(csv) else '⚠ CSV missing — run Phase 6 first')

# 6. Files
for f in ['pipeline/__init__.py','pipeline/dfd_parser.py','pipeline/prompt_templates.py','pipeline/response_parser.py','pipeline/inference.py']:
    p=f'/kaggle/working/SecureByDesign/{f}'
    print(f'{"✅" if os.path.exists(p) else "❌"} {f}')

print('\n✅ HANDOFF COMPLETE — Pipeline ready for Person B')

In [ ]:
import subprocess

REPO_URL = 'https://github.com/YOUR_USERNAME/SecureByDesign.git'  # ← CHANGE THIS
BASE = '/kaggle/working/SecureByDesign'

cmds = [
    f'git -C {BASE} init',
    f'git -C {BASE} config user.email "personA@securebydesign.dev"',
    f'git -C {BASE} config user.name "Person A"',
    f'git -C {BASE} remote add origin {REPO_URL} 2>/dev/null || git -C {BASE} remote set-url origin {REPO_URL}',
    f'git -C {BASE} add pipeline/ requirements.txt',
    f'git -C {BASE} commit -m "Person A: Pipeline complete. Phases 1-7 done. analyze_dfd() ready for Person B."',
    f'git -C {BASE} push -u origin main',
]
for cmd in cmds:
    r=subprocess.run(cmd,shell=True,capture_output=True,text=True)
    print(f'$ {cmd.split("-C")[1].strip()[:60]}')
    if r.stdout: print(' ',r.stdout.strip()[:100])
    if r.returncode!=0 and r.stderr: print(f'  NOTE: {r.stderr.strip()[:120]}')
print('\n✅ Git push complete.')